# Setup

## Install dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !uv pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install -q evaluate sacrebleu rouge_score bert_score nltk

## Configuration

In [ ]:
# Model settings (load from SFT checkpoint)
SFT_MODEL = "quannguyen204/vimed-llama3.2-3b-sft-v1"  
MAX_SEQ_LENGTH = 1500
DTYPE = None  # Auto-detect
LOAD_IN_4BIT = True

# LoRA settings (same as SFT for consistency)
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# DPO Training settings
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 2
LEARNING_RATE = 5e-6  # Lower LR for DPO
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
DPO_BETA = 0.1  # KL penalty coefficient

# Paths & Hub
DATASET_PATH = "quannguyen204/medical_dpo_synthetic_vi_3.6k_v1"  
OUTPUT_DIR = "./outputs/dpo"
HUB_MODEL_ID = "quannguyen204/vimed-llama3.2-3b-dpo-v1"

# W&B
WANDB_PROJECT = "ViMed-Assistant-DPO"
WANDB_RUN_NAME = "dpo-llama3.2-3b-med3.6k-v1"

## Authentication

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# HuggingFace login
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# W&B login
wandb_key = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_key

import wandb
wandb.login()

## Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"SFT Model loaded: {SFT_MODEL}")
print(f"Vocab size: {len(tokenizer)}")

## Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

## Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_PATH)

print(f"Dataset splits: {list(dataset.keys())}")
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"Columns: {dataset['train'].column_names}")

print("\n--- Sample ---")
print(f"Prompt: {dataset['train'][0]['prompt'][:200]}...")
print(f"Chosen: {dataset['train'][0]['chosen'][:200]}...")
print(f"Rejected: {dataset['train'][0]['rejected'][:200]}...")

## Prepare Dataset

In [ ]:
SYSTEM_PROMPT = "Bạn là một trợ lý y tế ảo thông minh, với vai trò là một bác sĩ tư vấn trực tuyến chuyên nghiệp và tận tâm. Nhiệm vụ của bạn là giải đáp thắc mắc, câu hỏi về chủ đề y tế. Câu trả lời cần mang tính định hướng, giải thích nguyên nhân có thể, không được thay thế chẩn đoán của bệnh viện và phải luôn khuyên người dùng đến cơ sở y tế để có chẩn đoán chính xác."

def format_dpo_sample(example):
    """
    Format DPO sample with chat template.
    Input: {"prompt": str, "chosen": str, "rejected": str}
    Output: {"prompt": formatted_prompt, "chosen": formatted_chosen, "rejected": formatted_rejected}
    """
    # Build prompt with system + user message
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"]}
    ]
    
    # Apply chat template (without generation prompt for DPO)
    formatted_prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    return {
        "prompt": formatted_prompt,
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

# Apply formatting to all splits
train_formatted = dataset["train"].map(format_dpo_sample, remove_columns=dataset["train"].column_names)
val_formatted = dataset["validation"].map(format_dpo_sample, remove_columns=dataset["validation"].column_names)
test_formatted = dataset["test"].map(format_dpo_sample, remove_columns=dataset["test"].column_names)

print("\n--- Formatted Sample ---")
print(f"Prompt: {train_formatted[0]['prompt'][:300]}...")

## Split Datasets

In [ ]:

train_dataset = train_formatted
eval_dataset = val_formatted  
test_dataset = test_formatted

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples (for training): {len(eval_dataset)}")
print(f"Test samples (for final evaluation): {len(test_dataset)}")

## Setup DPO Trainer

In [ ]:
from unsloth import PatchDPOTrainer
from trl import DPOTrainer, DPOConfig

# IMPORTANT: Patch DPOTrainer before using
PatchDPOTrainer()

# DPO Config
training_args = DPOConfig(
    # Output
    output_dir=OUTPUT_DIR,
    
    # DPO specific
    beta=DPO_BETA, 
    loss_type="sigmoid",  
    
    # Training hyperparams
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    
    # Optimizer
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    
    # Precision
    fp16=True,
    bf16=False, 
    
    # Evaluation & Saving
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Logging
    logging_steps=10,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    
    # Dataset
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    
    # Misc
    seed=42,
    dataloader_num_workers=4,
)

## Initialize Trainer

In [ ]:
# ============================================================
# Initialize DPOTrainer
# ============================================================
# Initialize W&B
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": SFT_MODEL,
        "method": "DPO",
        "beta": DPO_BETA,
        "lora_r": LORA_R,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "max_seq_length": MAX_SEQ_LENGTH,
    }
)

# Create DPO trainer
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth handles reference model internally
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

print("DPOTrainer initialized")

## GPU Check

In [ ]:
# ============================================================
# GPU Memory Check Before Training
# ============================================================
import torch

def print_gpu_memory():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"GPU {i}: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved, {total:.2f}GB total")

print_gpu_memory()

## Training

In [ ]:
# ============================================================
# Training
# ============================================================
print("Starting DPO training...")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Total training samples: {len(train_dataset)}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"DPO Beta: {DPO_BETA}")

# Train
trainer_stats = trainer.train()

print("\n--- Training Complete ---")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training runtime: {trainer_stats.metrics['train_runtime']:.2f}s")

# Evaluation

In [ ]:
# ============================================================
# DPO Evaluation on Test Set (Unseen Data)
# ============================================================
# DPO-specific metrics:
# 1. Reward Accuracy: % of times chosen response has higher log-prob than rejected
# 2. Reward Margin: Average difference between chosen and rejected log-probs
# 3. Generation Quality: BLEU, ROUGE, BERTScore on generated responses vs chosen

import evaluate
import numpy as np
from tqdm import tqdm
import torch
import torch.nn.functional as F

# Load text generation metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("Evaluation metrics loaded successfully")

## Load Evaluation Metrics

In [ ]:
# ============================================================
# Compute Reward Accuracy on Test Set
# ============================================================
# Reward Accuracy: Measures if model assigns higher probability to chosen vs rejected

def compute_log_probs(model, tokenizer, prompt, response, max_length=MAX_SEQ_LENGTH):
    """Compute log probability of response given prompt."""
    # Concatenate prompt and response
    full_text = prompt + response + tokenizer.eos_token
    
    # Tokenize
    inputs = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to("cuda")
    
    # Get prompt length to mask
    prompt_tokens = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to("cuda")
    prompt_len = prompt_tokens.input_ids.shape[1]
    
    # Forward pass
    with torch.no_grad():
        outputs = model(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask
        )
        logits = outputs.logits
    
    # Compute log probs for response tokens only
    shift_logits = logits[:, prompt_len-1:-1, :]
    shift_labels = inputs.input_ids[:, prompt_len:]
    
    # Get log probs
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)
    
    # Sum log probs (or use mean for length normalization)
    total_log_prob = token_log_probs.sum().item()
    mean_log_prob = token_log_probs.mean().item()
    
    return total_log_prob, mean_log_prob

print("Log probability computation function defined")

## Compute Reward Accuracy

In [ ]:
# ============================================================
# Evaluate Reward Accuracy on Test Set
# ============================================================
from unsloth import FastLanguageModel

# Keep model in training mode for evaluation (no .for_inference() yet)
model.eval()

# Sample subset for reward evaluation
NUM_REWARD_SAMPLES = min(200, len(test_dataset))
reward_indices = np.random.choice(len(test_dataset), NUM_REWARD_SAMPLES, replace=False)
reward_subset = test_dataset.select(reward_indices)

chosen_wins = 0
reward_margins = []

print(f"Computing reward accuracy on {NUM_REWARD_SAMPLES} test samples...")

for sample in tqdm(reward_subset, desc="Computing rewards"):
    prompt = sample["prompt"]
    chosen = sample["chosen"]
    rejected = sample["rejected"]
    
    try:
        # Compute log probs for chosen and rejected
        chosen_log_prob, chosen_mean_log_prob = compute_log_probs(model, tokenizer, prompt, chosen)
        rejected_log_prob, rejected_mean_log_prob = compute_log_probs(model, tokenizer, prompt, rejected)
        
        # Use mean log prob (length normalized) for fair comparison
        if chosen_mean_log_prob > rejected_mean_log_prob:
            chosen_wins += 1
        
        reward_margins.append(chosen_mean_log_prob - rejected_mean_log_prob)
    except Exception as e:
        print(f"Error computing log probs: {e}")
        continue

# Compute metrics
reward_accuracy = chosen_wins / len(reward_margins) * 100
mean_reward_margin = np.mean(reward_margins)
std_reward_margin = np.std(reward_margins)

print(f"\nReward evaluation complete")

## Evaluate Reward Accuracy

In [ ]:
# Generate predictions
# With 85/10/5 split: ~180 test samples for DPO dataset
NUM_GEN_SAMPLES = min(150, len(test_dataset))  # Comprehensive evaluation (83% of test set)
gen_indices = np.random.choice(len(test_dataset), NUM_GEN_SAMPLES, replace=False)
gen_subset = test_dataset.select(gen_indices)

predictions = []
chosen_refs = []
rejected_refs = []

print(f"Generating responses for {NUM_GEN_SAMPLES} test samples...")

for sample in tqdm(gen_subset, desc="Generating responses"):
    try:
        pred = generate_response(sample["prompt"])
        predictions.append(pred)
        chosen_refs.append(sample["chosen"])
        rejected_refs.append(sample["rejected"])
    except Exception as e:
        print(f"Error generating: {e}")
        continue

print(f"\nGenerated {len(predictions)} responses")

## Generate Responses

In [ ]:
# ============================================================
# Compute Generation Quality Metrics
# ============================================================
print("Computing generation quality metrics...")

# Compare generated responses vs CHOSEN responses (preferred)
# Higher similarity = model learned to generate preferred responses

# BLEU Score
bleu_vs_chosen = bleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in chosen_refs]
)

bleu_vs_rejected = bleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in rejected_refs]
)

# ROUGE Scores
rouge_vs_chosen = rouge_metric.compute(
    predictions=predictions,
    references=chosen_refs
)

rouge_vs_rejected = rouge_metric.compute(
    predictions=predictions,
    references=rejected_refs
)

# BERTScore (vs chosen only, more important)
bertscore_vs_chosen = bertscore_metric.compute(
    predictions=predictions,
    references=chosen_refs,
    lang="vi",
    model_type="bert-base-multilingual-cased"
)

bertscore_f1_chosen = np.mean(bertscore_vs_chosen["f1"])

print("Generation quality metrics computed")

## Compute Generation Quality

In [ ]:
# ============================================================
# Display DPO Evaluation Results
# ============================================================
print("\n" + "=" * 60)
print("DPO MODEL EVALUATION RESULTS (Test Set)")
print("=" * 60)

print("\nPREFERENCE ALIGNMENT METRICS:")
print(f"   Reward Accuracy: {reward_accuracy:.2f}%")
print("   (% of times model prefers chosen over rejected, higher is better)")
print(f"   Mean Reward Margin: {mean_reward_margin:.4f} ± {std_reward_margin:.4f}")
print("   (Positive = model prefers chosen responses)")

print("\nGENERATION QUALITY (vs Chosen Responses):")
print(f"   BLEU (vs Chosen):   {bleu_vs_chosen['score']:.2f}")
print(f"   BLEU (vs Rejected): {bleu_vs_rejected['score']:.2f}")
print(f"   → BLEU Preference Gap: {bleu_vs_chosen['score'] - bleu_vs_rejected['score']:.2f}")

print(f"\n   ROUGE-L (vs Chosen):   {rouge_vs_chosen['rougeL']:.4f}")
print(f"   ROUGE-L (vs Rejected): {rouge_vs_rejected['rougeL']:.4f}")
print(f"   → ROUGE-L Preference Gap: {rouge_vs_chosen['rougeL'] - rouge_vs_rejected['rougeL']:.4f}")

print(f"\n   BERTScore F1 (vs Chosen): {bertscore_f1_chosen:.4f}")
print("   (Semantic similarity to preferred responses)")

print("\n" + "=" * 60)

# Log to W&B
dpo_eval_metrics = {
    # Preference metrics
    "test/reward_accuracy": reward_accuracy,
    "test/mean_reward_margin": mean_reward_margin,
    "test/std_reward_margin": std_reward_margin,
    # Generation quality vs Chosen
    "test/bleu_vs_chosen": bleu_vs_chosen['score'],
    "test/rouge1_vs_chosen": rouge_vs_chosen['rouge1'],
    "test/rougeL_vs_chosen": rouge_vs_chosen['rougeL'],
    "test/bertscore_f1_vs_chosen": bertscore_f1_chosen,
    # Preference gap (higher = better alignment)
    "test/bleu_preference_gap": bleu_vs_chosen['score'] - bleu_vs_rejected['score'],
    "test/rougeL_preference_gap": rouge_vs_chosen['rougeL'] - rouge_vs_rejected['rougeL'],
    "test/num_reward_samples": len(reward_margins),
    "test/num_gen_samples": len(predictions),
}
wandb.log(dpo_eval_metrics)

print("\nMetrics logged to W&B")

## Display Results

In [ ]:
# ============================================================
# Sample Comparisons: Generated vs Chosen vs Rejected
# ============================================================
print("\n" + "=" * 60)
print("SAMPLE COMPARISONS")
print("=" * 60)

# Show 2 random examples
sample_indices = np.random.choice(len(predictions), min(2, len(predictions)), replace=False)

for i, idx in enumerate(sample_indices):
    print(f"\n{'─' * 60}")
    print(f"Example {i+1}:")
    print(f"{'─' * 60}")
    print(f"CHOSEN (Preferred):\n{chosen_refs[idx][:400]}...")
    print(f"\nREJECTED:\n{rejected_refs[idx][:400]}...")
    print(f"\nMODEL GENERATED:\n{predictions[idx][:400]}...")
    print()

## Sample Comparisons

In [ ]:
# ============================================================
# Save LoRA Adapter
# ============================================================
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

# Save & Upload

In [ ]:
# ============================================================
# Save Merged 16-bit Model for VLLM Serving
# ============================================================
# Merge LoRA weights with base model and save as 16-bit
# This is required for VLLM serving

model.save_pretrained_merged(
    f"{OUTPUT_DIR}/merged_16bit",
    tokenizer,
    save_method="merged_16bit",
)

print(f"Merged 16-bit model saved to {OUTPUT_DIR}/merged_16bit")

## Save LoRA Adapter

In [ ]:
# ============================================================
# Push to HuggingFace Hub
# ============================================================
# Push LoRA adapter
model.push_to_hub(
    HUB_MODEL_ID,
    token=hf_token,
    private=False,
)
tokenizer.push_to_hub(
    HUB_MODEL_ID,
    token=hf_token,
)

print(f"LoRA adapter pushed to https://huggingface.co/{HUB_MODEL_ID}")

# Push merged model for VLLM (takes longer)
model.push_to_hub_merged(
    f"{HUB_MODEL_ID}-merged",
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

print(f"Merged model pushed to https://huggingface.co/{HUB_MODEL_ID}-merged")

## Save Merged Model

In [ ]:
# ============================================================
# Inference Test - Compare with SFT
# ============================================================
from unsloth import FastLanguageModel

# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "Vitamin C có vai trò gì trong cơ thể?",
    "Tôi bị đau đầu và buồn nôn liên tục, tôi nên làm gì?",
    "Làm sao để nhận biết bệnh tiểu đường?"
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1500,
        temperature = 1.5, 
        min_p = 0.1,
        do_sample=True,
        use_cache=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"A: {response}")

## Push to HuggingFace Hub

In [ ]:
# ============================================================
# Cleanup
# ============================================================
wandb.finish()

print("\n" + "="*60)
print("DPO Training Complete!")
print(f"Model: {HUB_MODEL_ID}")
print(f"Merged Model: {HUB_MODEL_ID}-merged")
print("="*60)

## Inference Test